## [C] Convert Data to Long Format and Merge Data

In [1]:
import pandas as pd
import numpy as np
import os
from typing import Iterable, Union

In [2]:
path = "data/processed_data/"
file_name = "02_parameters.xlsx"

date_column = "date" 
stock_column = "stock"

# Set path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [4]:
presence_matrix_wide = pd.read_excel(path + file_name, sheet_name="presence_matrix_daily")
stock_returns_quarterly_wide = pd.read_excel(path + file_name, sheet_name="stock_returns_quarterly")
beta_wide = pd.read_excel(path + file_name, sheet_name="betas_quarterly")
size_wide = pd.read_excel(path + file_name, sheet_name="size_quarterly")
value_wide = pd.read_excel(path + file_name, sheet_name="value_quarterly")
profitability_wide = pd.read_excel(path + file_name, sheet_name="profitability_quarterly")
investment_wide = pd.read_excel(path + file_name, sheet_name="investment_quarterly")
price_momentum_wide = pd.read_excel(path + file_name, sheet_name="price_momentum_quarterly")
esg_score_quarterly_wide = pd.read_excel(path + file_name, sheet_name="esg_ff_quarterly")
esg_momentum_quarterly_wide = pd.read_excel(path + file_name, sheet_name="esg_momentum_quarterly")
industry_long = pd.read_excel(path + file_name, sheet_name="industry")
country_long = pd.read_excel(path + file_name, sheet_name="country")

In [5]:
stock_returns_monthly_wide = pd.read_excel(path + file_name, sheet_name="stock_returns_monthly")
esg_score_monthly_wide = pd.read_excel(path + file_name, sheet_name="esg_ff_monthly")
esg_momentum_monthly_wide = pd.read_excel(path + file_name, sheet_name="esg_momentum_monthly")
mcap_monthly_wide = pd.read_excel(path + file_name, sheet_name="outstanding_mcap_monthly")

fama_french_data_monthly = pd.read_excel(path + file_name, sheet_name="fama_french_data_monthly")
dividend_yield_monthly = pd.read_excel(path + file_name, sheet_name="dividend_yield_monthly")
term_spread_monthly = pd.read_excel(path + file_name, sheet_name="term_spread_monthly")
default_spread_monthly = pd.read_excel(path + file_name, sheet_name="default_spread_monthly")
yield_rate_monthly = pd.read_excel(path + file_name, sheet_name="yield_monthly")
recession_indicator = pd.read_excel(path + file_name, sheet_name="recession_indicator_monthly")

## 1. Prepare Data for Analysis

In [6]:
def winsorize_by_quarter_inplace(
    df_raw,
    date_col = "date",
    cols = "Return",
    p = 0.005,
    occurrance = "Q"):
    """
    Winsorize one or multiple columns within each quarter based on date_col,
    replacing the original column values (same column names).
    """

    df = df_raw.copy()

    df[date_col] = pd.to_datetime(df[date_col])
    q = df[date_col].dt.to_period(occurrance)

    # normalize cols to list
    cols_list = [cols] if isinstance(cols, str) else list(cols)

    for c in cols_list:
        s = pd.to_numeric(df[c], errors="coerce")
        lower = s.groupby(q).transform(lambda v: v.quantile(p))
        upper = s.groupby(q).transform(lambda v: v.quantile(1 - p))
        df[c] = s.clip(lower=lower, upper=upper)

    return df

In [ ]:
def transform_data_to_long_format(wide_df, value_name):
    """
    Transforms a wide-format DataFrame to a long-format DataFrame.
    """

    # Convert the DataFrame to long format
    long_df = pd.melt(wide_df, id_vars=[date_column], var_name=stock_column, value_name=value_name)

    long_df[date_column] = pd.to_datetime(long_df[date_column]).dt.date

    return long_df

In [8]:
def getPresenceMatrix(data, quarterly):
    """Transforms the presence matrix to a quarterly or monthly format, keeping only the last observation in each period."""
    presence_matrix = data.copy()

    # Sort by date and set date as index for resampling
    presence_matrix = presence_matrix.sort_values(date_column).set_index(date_column)

    # Resample the data to quarterly or monthly frequency, taking the last observation in each period
    if quarterly:
        presence_matrix = presence_matrix.resample("QE").last()
    else:
        presence_matrix = presence_matrix.resample("ME").last()

    # Drop rows where all values are NaN and reset the index
    presence_matrix = presence_matrix.dropna(how="all").reset_index().rename(columns={"index": date_column})

    # Convert the date column to datetime and then to date format
    presence_matrix[date_column] = presence_matrix[date_column].dt.date

    return presence_matrix

In [9]:
presence_matrix_wide_quarterly = getPresenceMatrix(presence_matrix_wide, quarterly=True)
presence_matrix_wide_monthly = getPresenceMatrix(presence_matrix_wide, quarterly=False)

In [10]:
presence_matrix_wide_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2016-06-30,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
1,2016-09-30,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
2,2016-12-31,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,0,1
3,2017-03-31,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,0,1
4,2017-06-30,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,0,1


In [11]:
presence_matrix_wide_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2016-06-30,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
1,2016-07-31,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
2,2016-08-31,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
3,2016-09-30,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
4,2016-10-31,1,0,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1


### POLS, FE, RE, GMM Data

In [12]:
presence_matrix_quarterly_long = transform_data_to_long_format(presence_matrix_wide_quarterly, "presence")
stock_returns_quarterly_long = transform_data_to_long_format(stock_returns_quarterly_wide, "stock_return_quarterly")
beta_long = transform_data_to_long_format(beta_wide, "beta")
size_long = transform_data_to_long_format(size_wide, "size")
value_long = transform_data_to_long_format(value_wide, "value")
profitability_long = transform_data_to_long_format(profitability_wide, "profitability")
investment_long = transform_data_to_long_format(investment_wide, "investment")
price_momentum_long = transform_data_to_long_format(price_momentum_wide, "price_momentum")
esg_score_quarterly_long = transform_data_to_long_format(esg_score_quarterly_wide, "esg_score")
esg_momentum_quarterly_long = transform_data_to_long_format(esg_momentum_quarterly_wide, "esg_momentum")

In [13]:
# Merge all long-format DataFrames on date and stock columns
merged_pols_fe_re_gmm_data = presence_matrix_quarterly_long.merge(stock_returns_quarterly_long, on=[date_column, stock_column], how="outer") \
    .merge(beta_long, on=[date_column, stock_column], how="outer") \
    .merge(size_long, on=[date_column, stock_column], how="outer") \
    .merge(value_long, on=[date_column, stock_column], how="outer") \
    .merge(profitability_long, on=[date_column, stock_column], how="outer") \
    .merge(investment_long, on=[date_column, stock_column], how="outer") \
    .merge(price_momentum_long, on=[date_column, stock_column], how="outer") \
    .merge(esg_score_quarterly_long, on=[date_column, stock_column], how="outer") \
    .merge(esg_momentum_quarterly_long, on=[date_column, stock_column], how="outer")\
    .merge(industry_long, on=[stock_column], how="left") \
    .merge(country_long, on=[stock_column], how="left")

In [26]:
# Filter to only include rows where presence == 1
pols_fe_re_gmm_data = merged_pols_fe_re_gmm_data[merged_pols_fe_re_gmm_data["presence"] == 1].drop(columns=["presence"])
pols_fe_re_gmm_data[date_column] = pd.to_datetime(pols_fe_re_gmm_data[date_column]).dt.date

initial_row_count = pols_fe_re_gmm_data.shape[0]

print("There are {} rows and {} columns in the merged dataset.\n".format(pols_fe_re_gmm_data.shape[0], pols_fe_re_gmm_data.shape[1]))

# Drop rows with any NaN values
pols_fe_re_gmm_data = pols_fe_re_gmm_data.dropna(subset=["stock_return_quarterly", "beta", "size", "value", "profitability", "investment", "price_momentum", "esg_score", "esg_momentum", "industry", "country_of_headquarters"])

print("{} rows are dropped after removing rows with any NaN values.".format(initial_row_count - pols_fe_re_gmm_data.shape[0]))
print("There are {} rows remaining after dropping rows with NaN values.\n".format(pols_fe_re_gmm_data.shape[0]))

# Export non winsorized data for summary stats
pols_fe_re_gmm_data.to_csv(path + "03_pols_fe_re_gmm_data.csv", index=False)

pols_fe_re_gmm_data.head()

There are 22770 rows and 13 columns in the merged dataset.

2586 rows are dropped after removing rows with any NaN values.
There are 20184 rows remaining after dropping rows with NaN values.



,date,stock,stock_return_quarterly,beta,size,value,profitability,investment,price_momentum,esg_score,esg_momentum,industry,country_of_headquarters
2,2016-06-30,A2.MI,0.030621,1.470557,22.009083,0.748281,0.079054,-0.033443,-0.036842,59.459100,-1.622974,Utilities,Italy
6,2016-06-30,AAL.L,0.250263,0.558792,23.225868,1.461129,-0.304424,0.000000,0.340393,81.434442,-7.452243,Basic Materials,United Kingdom
7,2016-06-30,AALB.AS,-0.112824,1.229154,21.818938,0.424145,0.194845,0.000000,0.014720,17.964003,5.607441,Industrials,Netherlands
8,2016-06-30,ABBN.S,0.029602,0.988284,24.365696,0.362603,0.200431,0.011075,0.041707,88.899243,-3.994408,Industrials,Switzerland
9,2016-06-30,ABDN.L,-0.217060,1.664075,22.662034,0.796699,0.150175,0.000000,-0.256856,74.424211,-0.391662,Financials,United Kingdom


In [15]:
columns_to_winsorize = ["beta", "size", "value", "profitability", "investment", "price_momentum", "esg_score", "esg_momentum"]

# Winsorize the specified columns by quarter, capping values at the 0.5% and 99.5% quantiles within each quarter
pols_fe_re_gmm_winsorized_data = winsorize_by_quarter_inplace(pols_fe_re_gmm_data, date_col=date_column, cols=columns_to_winsorize, p=0.005)                                                   

# Export data as CSV
pols_fe_re_gmm_winsorized_data.to_csv(path + "03_pols_fe_re_gmm_winsorized_data.csv", index=False)

In [16]:
pols_fe_re_gmm_winsorized_data.head()

,date,stock,stock_return_quarterly,beta,size,value,profitability,investment,price_momentum,esg_score,esg_momentum,industry,country_of_headquarters
2,2016-06-30,A2.MI,0.030621,1.470557,22.009083,0.748281,0.079054,-0.033443,-0.036842,59.459100,-1.622974,Utilities,Italy
6,2016-06-30,AAL.L,0.250263,0.558792,23.225868,1.461129,-0.304424,0.000000,0.324333,81.434442,-7.452243,Basic Materials,United Kingdom
7,2016-06-30,AALB.AS,-0.112824,1.229154,21.818938,0.424145,0.194845,0.000000,0.014720,17.964003,5.607441,Industrials,Netherlands
8,2016-06-30,ABBN.S,0.029602,0.988284,24.365696,0.362603,0.200431,0.011075,0.041707,88.899243,-3.994408,Industrials,Switzerland
9,2016-06-30,ABDN.L,-0.217060,1.664075,22.662034,0.796699,0.150175,0.000000,-0.256856,74.424211,-0.391662,Financials,United Kingdom


In [17]:
unique_per_date = (
    pols_fe_re_gmm_data.groupby("date")["stock"]
      .nunique()
      .reset_index(name="n_unique_stocks")
)

unique_per_date

,date,n_unique_stocks
0,2016-06-30,468
1,2016-09-30,493
2,2016-12-31,512
3,2017-03-31,512
4,2017-06-30,512
5,2017-09-30,509
6,2017-12-31,516
7,2018-03-31,514
8,2018-06-30,511
9,2018-09-30,506


### Business Cycle Analysis Data

In [18]:
presence_matrix_monthly_long = transform_data_to_long_format(presence_matrix_wide_monthly, "presence")
stock_returns_monthly_long = transform_data_to_long_format(stock_returns_monthly_wide, "stock_return")
esg_momentum_monthly_long = transform_data_to_long_format(esg_momentum_monthly_wide, "esg_momentum_score")
mcap_monthly_long = transform_data_to_long_format(mcap_monthly_wide, "mcap")

fama_french_data_monthly[date_column] = pd.to_datetime(fama_french_data_monthly[date_column]).dt.date   
dividend_yield_monthly[date_column] = pd.to_datetime(dividend_yield_monthly[date_column]).dt.date
term_spread_monthly[date_column] = pd.to_datetime(term_spread_monthly[date_column]).dt.date
default_spread_monthly[date_column] = pd.to_datetime(default_spread_monthly[date_column]).dt.date
yield_rate_monthly[date_column] = pd.to_datetime(yield_rate_monthly[date_column]).dt.date
recession_indicator[date_column] = pd.to_datetime(recession_indicator[date_column]).dt.date

In [19]:
merged_bc_data = presence_matrix_monthly_long.merge(stock_returns_monthly_long, on=[date_column, stock_column], how="outer") \
    .merge(esg_momentum_monthly_long, on=[date_column, stock_column], how="outer") \
    .merge(mcap_monthly_long, on=[date_column, stock_column], how="outer") \
    .merge(fama_french_data_monthly, on=[date_column], how="left") \
    .merge(dividend_yield_monthly, on=[date_column], how="left") \
    .merge(term_spread_monthly, on=[date_column], how="left") \
    .merge(default_spread_monthly, on=[date_column], how="left") \
    .merge(yield_rate_monthly, on=[date_column], how="left") \
    .merge(recession_indicator, on=[date_column], how="left")

merged_bc_data.drop(columns=["AAA", "BAA", "total_dividend_points", ".STOXX_prices", "total_dividend_points_ttm"], inplace=True)

In [20]:
merged_bc_data.head()

,date,stock,presence,stock_return,esg_momentum_score,mcap,Mkt-RF,SMB,HML,RMW,CMA,RF,WML,dividend_yield_ttm,term_spread,default_spread,spot_rate_3m_government_bond,recession_indicator
0,2016-06-30,1COVG.DE^L25,1.0,0.048150,NaN,8.088862e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
1,2016-06-30,1U1.DE,0.0,-0.064258,NaN,1.886095e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
2,2016-06-30,A2.MI,1.0,-0.080406,-1.622974,3.617623e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
3,2016-06-30,AAAA.L^C21,1.0,-0.232982,2.922803,1.749150e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
4,2016-06-30,AAF.L,0.0,NaN,NaN,NaN,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0


In [21]:
# Filter to only include rows where presence == 1
bc_data = merged_bc_data[merged_bc_data["presence"] == 1].drop(columns=["presence"])
bc_data[date_column] = pd.to_datetime(bc_data[date_column]).dt.date

initial_row_count_bc = bc_data.shape[0]

print("There are {} rows and {} columns in the merged dataset.\n".format(bc_data.shape[0], bc_data.shape[1]))

# Drop rows with any NaN values except for recession indicator (which is only used for one specific analysis)
ignore_col = "recession_indicator"

cols_to_check = bc_data.columns.difference([ignore_col])
bc_data_clean = bc_data.dropna(subset=cols_to_check)

print("{} rows are dropped after removing rows with any NaN values.".format(initial_row_count_bc - bc_data_clean.shape[0]))
print("There are {} rows remaining after dropping rows with NaN values.\n".format(bc_data_clean.shape[0]))

bc_data_clean.head()

There are 67115 rows and 17 columns in the merged dataset.

1574 rows are dropped after removing rows with any NaN values.
There are 65541 rows remaining after dropping rows with NaN values.



,date,stock,stock_return,esg_momentum_score,mcap,Mkt-RF,SMB,HML,RMW,CMA,RF,WML,dividend_yield_ttm,term_spread,default_spread,spot_rate_3m_government_bond,recession_indicator
2,2016-06-30,A2.MI,-0.080406,-1.622974,3.617623e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
3,2016-06-30,AAAA.L^C21,-0.232982,2.922803,1.749150e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
6,2016-06-30,AAL.L,0.115835,-7.452243,1.221423e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
7,2016-06-30,AALB.AS,-0.146957,5.607441,2.991192e+09,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0
8,2016-06-30,ABBN.S,-0.056730,-3.994408,3.818451e+10,-0.0501,-0.0224,-0.0164,0.0325,0.0282,0.0002,0.071,0.045862,0.005456,0.0103,-0.0065,0


In [22]:
# Export data as CSV
bc_data_clean.to_csv(path + "03_bc_data.csv", index=False)

In [25]:
unique_per_date = (
    bc_data_clean.groupby("date")["stock"]
      .nunique()
      .reset_index(name="n_unique_stocks")
)

unique_per_date.head(100)

,date,n_unique_stocks
0,2016-06-30,534
1,2016-07-31,538
2,2016-08-31,543
3,2016-09-30,560
4,2016-10-31,562
...,...,...
95,2024-05-31,593
96,2024-06-30,592
97,2024-07-31,592
98,2024-08-31,592
